## Project Title

Customer Behavior & Revenue Analysis for Churn Detection
(SQL | UK Online Retail Dataset)

## Project Overview

### Objective

This project aims to analyze transactional retail data using SQL to uncover key patterns in customer behavior and revenue generation, with a particular focus on identifying early signals of customer churn.

The analysis is designed to support data-driven decision-making in:

   - Customer retention strategies

   - Customer segmentation

   - Revenue optimization

Additionally, this SQL analysis serves as the foundational layer for downstream tasks, including RFM-based customer segmentation and churn prediction modeling implemented in Python.

### Dataset Description

The dataset contains real-world transactional records from a UK-based online retailer. Each row represents a product-level transaction within an invoice and includes:

   - Customer identifiers

   - Purchase timestamps

   - Product quantities

   - Unit prices

Data source:
- UCI Machine Learning Repository:
  https://archive.ics.uci.edu/ml/datasets/Online+Retail
  
This granular structure enables detailed analysis of:

   - Customer purchasing behavior

   - Customer lifecycle patterns

   - Revenue contribution across different customer segments

## 1. Data Ingestion & Initial Data Validation

In this step, raw transactional data is loaded and prepared for analysis. 
Initial data validation checks are performed to assess data quality, ensure correct data types, 
and identify potential issues (e.g., missing customer identifiers) that may impact downstream churn analysis.

This step is critical to ensure the reliability of subsequent customer behavior and revenue analysis.

In [1]:
import pandas as pd

# Read csv dataset
retail_df = pd.read_csv("data/online_retail.csv")

# Convert InvoiceDate to datetime format to enable time-based analysis.
retail_df["InvoiceDate"] = pd.to_datetime(retail_df["InvoiceDate"])

# Basic data quality checks
print(retail_df.shape)

# Note: Missing CustomerID values may indicate anonymous transactions 
# and will need to be handled before customer-level analysis (e.g., churn modeling).
print(retail_df.isnull().sum())
print(retail_df.dtypes)

(541909, 8)
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object


Ensured appropriate data types and schema compatibility before ingestion.

After initial validation, the dataset is loaded into a relational SQLite database to enable efficient querying and scalable analytical workflows.

Storing the data in a structured database allows for more flexible data exploration, aggregation, and feature extraction using SQL, which is essential for customer-level analysis and churn-related insights.

In [2]:
import sqlite3

# Create a connection to a SQLite database (or connect to an existing one)
conn = sqlite3.connect("data/raw_online_retail.db")

# Using a relational database enables efficient aggregation at the customer level,
# which is critical for downstream tasks such as RFM analysis and churn detection.
# Replace table if it already exists to ensure reproducibility of the pipeline
retail_df.to_sql(
    name="raw_online_retail",
    con=conn,
    if_exists="replace",
    index=False
)

# Optional: close the connection when done
#conn.close()

541909

## 2. Data Understanding & Initial Exploration

### 2.1. Schema Overview

Understanding the schema ensures that key fields required for customer-level analysis (e.g., CustomerID, InvoiceDate) are correctly structured and usable.

In [3]:
pd.read_sql("""
PRAGMA table_info(raw_online_retail);
""", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,InvoiceNo,TEXT,0,None,0
1,1,StockCode,TEXT,0,None,0
2,2,Description,TEXT,0,None,0
3,3,Quantity,INTEGER,0,None,0
4,4,InvoiceDate,TIMESTAMP,0,None,0
5,5,UnitPrice,REAL,0,None,0
6,6,CustomerID,REAL,0,None,0
7,7,Country,TEXT,0,None,0


### 2.2. Sample Data

A quick inspection of sample records helps validate data consistency and provides an initial understanding of transaction structure, including multi-item invoices and customer purchase patterns.

In [4]:
pd.read_sql("""
SELECT 
    * 
FROM raw_online_retail
LIMIT 10;
""", conn)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


### 2.3. Dataset Size

The dataset contains over 540K transaction records, providing a robust foundation for analyzing customer behavior and identifying churn patterns at scale.

In [5]:
pd.read_sql("""
SELECT 
    COUNT(*) AS total_rows 
FROM raw_online_retail;
""", conn)

,total_rows
0,541909


### 2.4. Missing Values Check

In [6]:
pd.read_sql("""
SELECT
    COUNT(*) AS total,
    COUNT(*) - COUNT(CustomerID) AS missing,
    ROUND(100.0 * (COUNT(*) - COUNT(CustomerID)) / COUNT(*), 2) AS missing_pct 
FROM raw_online_retail;
""", conn)

,total,missing,missing_pct
0,541909,135080,24.93




Approximately 25% of transactions lack CustomerID, which prevents linking these records to individual customers.

Since churn analysis requires customer-level tracking, these records will be excluded from downstream analysis. This highlights a key data limitation and ensures that subsequent insights are based on reliable customer data.

### 2.5. Negative / Invalid Values

Transactions with negative quantities or non-positive prices likely represent returns or cancellations.

To ensure accurate revenue analysis, these records will be handled separately and excluded from revenue-related calculations. However, they may still provide useful signals for customer behavior (e.g., dissatisfaction).

In [7]:
pd.read_sql("""
SELECT 
    * 
FROM raw_online_retail 
WHERE Quantity <= 0 OR UnitPrice <= 0;

""", conn)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
1,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
2,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
3,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
4,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
...,...,...,...,...,...,...,...,...
11800,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom
11801,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom
11802,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom
11803,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom


### 2.6. Dataset Summary Statistics

In [8]:
pd.read_sql("""
SELECT 
    COUNT(*) AS total_rows,
    COUNT(DISTINCT InvoiceNo) AS total_invoices,
    COUNT(DISTINCT CustomerID) AS total_customers,
    COUNT(DISTINCT Country) AS total_countries
FROM raw_online_retail;
""", conn)

,total_rows,total_invoices,total_customers,total_countries
0,541909,25900,4372,38




The dataset includes over 500K transactions, ~25K invoices, and ~4.3K unique customers across 38 countries.

This indicates a diverse and active customer base suitable for behavioral segmentation.

However, the presence of missing customer identifiers introduces a limitation that must be addressed during data cleaning to ensure accurate customer-level insights.

## 3. Data Quality & Cleaning

To ensure reliable customer-level analysis, a cleaned analytical dataset is constructed by removing transactions that do not represent genuine customer purchases.

This includes:
- Cancelled transactions (returns/refunds)
- Invalid records (non-positive quantities or prices)
- Transactions without customer identifiers

This step is critical to ensure that downstream analyses—such as revenue estimation, RFM segmentation, and churn modeling—are based on accurate and meaningful data.

### 3.1. Remove cancellations

Cancelled transactions are identified by InvoiceNo values starting with 'C', which typically represent returns or refunds.


In [9]:
pd.read_sql("""
SELECT *
FROM raw_online_retail
WHERE InvoiceNo LIKE 'C%'
LIMIT 10;
""", conn)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
1,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
2,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
3,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
4,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
5,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
6,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
7,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
8,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
9,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


In [10]:
pd.read_sql("""
SELECT 
    COUNT(*) AS total_rows,
    SUM(CASE WHEN InvoiceNo LIKE 'C%' THEN 1 ELSE 0 END) AS cancelled,
    ROUND(100.0 * SUM(CASE WHEN InvoiceNo LIKE 'C%' THEN 1 ELSE 0 END) / COUNT(*), 2) AS cancelled_pct
FROM raw_online_retail;
""", conn)

,total_rows,cancelled,cancelled_pct
0,541909,9288,1.71


Cancelled invoices (identified by InvoiceNo starting with 'C') represent returned or refunded transactions and do not reflect actual sales. 

Since these transactions do not reflect actual purchases, including them would distort revenue calculations and customer behavior analysis.

Approximately 1.7% of transactions are cancellations. While relatively small, they must be excluded to ensure analytical accuracy.

These transactions may also provide useful signals about customer dissatisfaction and could be explored separately as a potential indicator of churn.

### 3.2. Create a clean CTE 
Here, a CTE (Common Table Expression) is created, which is a temporary, reusable query result that helps simplify complex SQL queries.

In [11]:
## Define a clean analytical dataset by applying all filtering rules in a single, reusable step
pd.read_sql("""
WITH clean_online_retail AS (
    SELECT *
    FROM raw_online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
      AND CustomerID IS NOT NULL
)
SELECT * FROM clean_online_retail
""", conn)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
397879,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
397880,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
397881,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
397882,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


A cleaned analytical dataset is constructed using a Common Table Expression (CTE), enabling a clear and reusable definition of data filtering logic.

The filtering criteria ensure that only valid, customer-linked purchase transactions are retained for analysis.

This structured approach improves query readability, reproducibility, and scalability of the analysis pipeline.

## 4. Create a revenue column

### 4.1. Defining Revenue Metric

Revenue is defined at the transaction level as the product of Quantity and UnitPrice.

This definition captures the monetary value of each purchased item and serves as the foundation for all revenue-based analyses, including customer lifetime value estimation, segmentation, and churn-related insights.

##### Key assumptions for revenue calculation:

- Revenue is computed as Quantity × UnitPrice at the transaction level.
- Cancelled transactions (InvoiceNo starting with 'C') are excluded, as they represent returns or refunds.
- Transactions with non-positive quantities or prices are considered invalid for revenue analysis.
- Only transactions with valid CustomerID are included to enable customer-level aggregation.

These assumptions ensure that revenue metrics accurately reflect actual customer purchasing behavior.

In [12]:
pd.read_sql("""
WITH clean_online_retail AS (
    SELECT *
    FROM raw_online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
      AND CustomerID IS NOT NULL
)
SELECT *,
       Quantity * UnitPrice AS revenue
FROM clean_online_retail;
""", conn)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
...,...,...,...,...,...,...,...,...,...
397879,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,10.20
397880,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,12.60
397881,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60
397882,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60


Revenue is first computed at the transaction level and will later be aggregated at the customer level to support behavioral segmentation and churn analysis.

## 5. Core Business Analysis

### 5.1. Overall Revenue Analysis
The first business metric analyzed is total revenue, which provides a high-level view of the company’s sales performance over the observed time period.

In [13]:
pd.read_sql("""
WITH clean_online_retail AS (
    SELECT *
    FROM raw_online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
      AND CustomerID IS NOT NULL
)
SELECT
    ROUND(SUM(Quantity * UnitPrice), 2) AS total_revenue
FROM clean_online_retail;
""", conn)

,total_revenue
0,8911407.9


Total revenue exceeds €8.9M over the observed period, indicating strong overall sales performance.

This metric establishes a baseline for evaluating business growth and enables further analysis of revenue distribution across time, customers, and geographic regions.

However, aggregate revenue alone does not reveal underlying behavioral patterns, making deeper segmentation essential for actionable insights.

### 5.2 Monthly revenue trend
To identify temporal patterns and seasonality, I analyze revenue aggregated at a monthly level.

In [14]:
pd.read_sql("""
WITH clean_online_retail AS (
    SELECT *,
           Quantity * UnitPrice AS revenue,
           InvoiceDate AS invoice_ts
    FROM raw_online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
      AND CustomerID IS NOT NULL
)
SELECT
    strftime('%Y-%m', invoice_ts) AS month,
    ROUND(SUM(revenue), 2) AS monthly_revenue
FROM clean_online_retail
GROUP BY month
ORDER BY month;
""", conn)

,month,monthly_revenue
0,2010-12,572713.89
1,2011-01,569445.04
2,2011-02,447137.35
3,2011-03,595500.76
4,2011-04,469200.36
5,2011-05,678594.56
6,2011-06,661213.69
7,2011-07,600091.01
8,2011-08,645343.90
9,2011-09,952838.38


Monthly revenue shows a clear upward trend throughout 2011, peaking in November at approximately €1.16M.

This pattern suggests strong seasonality, likely driven by pre-holiday shopping behavior.

The sharp decline in December may indicate incomplete data or post-peak normalization, and should be interpreted with caution.

These temporal patterns are critical for forecasting demand and planning marketing or inventory strategies.

### 5.3 Country-level performance

To understand geographic revenue distribution, I analyze total revenue by country and calculate each country's contribution to overall sales.

This helps identify key markets, assess revenue concentration, and uncover opportunities for international growth.

In [15]:
pd.read_sql("""
WITH clean_online_retail AS (
    SELECT *,
           Quantity * UnitPrice AS revenue,
           InvoiceDate AS invoice_ts
    FROM raw_online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
      AND CustomerID IS NOT NULL
),
country_revenue AS (
    SELECT
        Country,
        SUM(revenue) AS country_revenue
    FROM clean_online_retail
    GROUP BY Country
)
SELECT
    Country,
    ROUND(country_revenue, 2) AS country_revenue,
    ROUND(
        country_revenue * 100.0 / SUM(country_revenue) OVER (),
        2
    ) AS contribution_pct
FROM country_revenue
ORDER BY country_revenue DESC;
""", conn)

,Country,country_revenue,contribution_pct
0,United Kingdom,7308391.55,82.01
1,Netherlands,285446.34,3.20
2,EIRE,265545.90,2.98
3,Germany,228867.14,2.57
4,France,209024.05,2.35
5,Australia,138521.31,1.55
6,Spain,61577.11,0.69
7,Switzerland,56443.95,0.63
8,Belgium,41196.34,0.46
9,Sweden,38378.33,0.43


Revenue is highly concentrated in the United Kingdom, accounting for approximately 82% of total sales.

In contrast, all other countries individually contribute less than 4%, indicating a significant imbalance in geographic revenue distribution.

This suggests a strong dependence on the domestic market, which may pose a risk in terms of revenue diversification.

At the same time, it highlights potential opportunities for international expansion and targeted growth strategies in underpenetrated markets.

This geographic imbalance may also influence churn behavior, as customer engagement and retention dynamics can vary significantly across regions.

## 6. Product analytics (business-relevant)

### 6.1. Customer Lifetime Value Analysis

To better understand how revenue is distributed across customers, the customer lifetime value (CLV) is calculated as the total revenue generated by each customer over the observed time period. CLV helps identify high-value customers and assess the degree of revenue concentration, which is critical for customer retention and targeted marketing strategies.

In [16]:
pd.read_sql("""
WITH clean_online_retail AS (
    SELECT *,
           Quantity * UnitPrice AS revenue,
           InvoiceDate AS invoice_ts
    FROM raw_online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
      AND CustomerID IS NOT NULL
) 
SELECT 
    CustomerID,
    ROUND(SUM(revenue), 2) AS lifetime_value
FROM clean_online_retail
GROUP BY CustomerID
ORDER BY lifetime_value DESC;   
""", conn)

,CustomerID,lifetime_value
0,14646.0,280206.02
1,18102.0,259657.30
2,17450.0,194550.79
3,16446.0,168472.50
4,14911.0,143825.06
...,...,...
4333,16878.0,13.30
4334,17956.0,12.75
4335,16454.0,6.90
4336,14792.0,6.20


Customer lifetime value (CLV) analysis reveals a highly skewed revenue distribution, where a small group of customers generates a disproportionately large share of total revenue.

This indicates that the business relies heavily on high-value customers, making their retention critical for sustained revenue growth.

Identifying and prioritizing these customers enables targeted strategies such as personalized offers, loyalty programs, and proactive engagement to reduce churn risk.

### 6.2 Purchase Frequency

To complement customer lifetime value, the purchase frequency is analysed by measuring how often each customer places an order. Purchase frequency helps distinguish between customers who generate high revenue through repeated purchases and those who contribute mainly through high-value single transactions. This provides additional insight into customer engagement and loyalty.

In [17]:
pd.read_sql("""
WITH clean_online_retail AS (
    SELECT *,
           Quantity * UnitPrice AS revenue,
           InvoiceDate AS invoice_ts
    FROM raw_online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
      AND CustomerID IS NOT NULL
) 
SELECT
    CustomerID,
    COUNT(DISTINCT InvoiceNo) AS num_orders,
    ROUND(SUM(revenue), 2) AS total_spend
FROM clean_online_retail
GROUP BY CustomerID
ORDER BY num_orders DESC;  
""", conn)

,CustomerID,num_orders,total_spend
0,12748.0,209,33719.73
1,14911.0,201,143825.06
2,17841.0,124,40991.57
3,13089.0,97,58825.83
4,14606.0,93,12156.65
...,...,...,...
4333,12354.0,1,1079.40
4334,12353.0,1,89.00
4335,12350.0,1,334.40
4336,12349.0,1,1757.55


Customers with higher purchase frequency contribute significantly more to total revenue, indicating a strong relationship between repeat purchasing behavior and customer value.

At the same time, a large portion of customers make only a single purchase, highlighting a gap between engaged (loyal) customers and one-time buyers.

This suggests an opportunity to improve customer retention by converting one-time buyers into repeat customers through targeted engagement strategies.

### 6.3 Repeat vs One-Time Customers

##### Customer Retention Analysis
To further understand customer behavior, the customers are classified into one-time and repeat buyers based on the number of distinct invoices associated with each customer. This analysis helps assess customer retention and identify whether revenue is primarily driven by long-term relationships or single-purchase customers.

In [18]:
pd.read_sql("""
WITH clean_online_retail AS (
    SELECT *,
           Quantity * UnitPrice AS revenue,
           InvoiceDate AS invoice_ts
    FROM raw_online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
      AND CustomerID IS NOT NULL
) 
SELECT 
    customer_type,
    COUNT(*) AS customers,
    ROUND(SUM(total_spend), 2) AS total_revenue
FROM (
    SELECT 
        CustomerID,
        COUNT(DISTINCT InvoiceNo) AS num_orders,
        SUM(Quantity * UnitPrice) AS total_spend,
        CASE 
            WHEN COUNT(DISTINCT InvoiceNo) = 1 THEN 'One-time'
            ELSE 'Repeat'
        END AS customer_type
    FROM clean_online_retail
    GROUP BY CustomerID
)
GROUP BY customer_type;
""", conn)

,customer_type,customers,total_revenue
0,One-time,1493,616311.73
1,Repeat,2845,8295096.17


#### One-time:

   - 1493 customers (~35%)

   - €616K revenue (~7%)

#### Repeat:

   - 2845 customers (~65%)

   - €8.29M revenue (~93%) 💥

Repeat customers account for approximately 65% of the customer base but generate over 93% of total revenue, while one-time customers contribute only a small fraction of revenue.

This highlights that long-term customer relationships are the primary drivers of business performance.

The low contribution from one-time customers suggests potential inefficiencies in customer acquisition or onboarding, and emphasizes the need for strategies focused on early-stage retention and engagement.

Improving the conversion rate from one-time to repeat customers could have a significant impact on overall revenue growth.

## 7. Pareto Analysis (Revenue Concentration)

To evaluate revenue concentration, a Pareto analysis is performed by ranking customers based on their total revenue contribution and calculating the cumulative share of revenue.

This helps identify whether a small group of customers drives the majority of sales.

In [19]:
pd.read_sql("""
WITH clean_online_retail AS (
    SELECT *,
           Quantity * UnitPrice AS revenue,
           InvoiceDate AS invoice_ts
    FROM raw_online_retail
    WHERE InvoiceNo NOT LIKE 'C%'
      AND Quantity > 0
      AND UnitPrice > 0
      AND CustomerID IS NOT NULL
),
customer_revenue AS (
    SELECT
        CustomerID,
        SUM(revenue) AS total_revenue
    FROM clean_online_retail
    GROUP BY CustomerID
),
ranked_customers AS (
    SELECT
        CustomerID,
        total_revenue,
        SUM(total_revenue) OVER (
            ORDER BY total_revenue DESC
        ) AS cumulative_revenue,
        SUM(total_revenue) OVER () AS overall_revenue
    FROM customer_revenue
)
SELECT
    CustomerID,
    ROUND(total_revenue, 2) AS customer_revenue,
    ROUND((cumulative_revenue / overall_revenue) * 100, 2) AS cumulative_revenue_pct
FROM ranked_customers
ORDER BY customer_revenue DESC;
""", conn)

,CustomerID,customer_revenue,cumulative_revenue_pct
0,14646.0,280206.02,3.14
1,18102.0,259657.30,6.06
2,17450.0,194550.79,8.24
3,16446.0,168472.50,10.13
4,14911.0,143825.06,11.75
...,...,...,...
4333,16878.0,13.30,100.00
4334,17956.0,12.75,100.00
4335,16454.0,6.90,100.00
4336,14792.0,6.20,100.00


The Pareto analysis confirms that a small proportion of customers contributes a disproportionately large share of total revenue.

This concentration indicates a high dependency on a limited number of high-value customers, which introduces potential revenue risk if these customers churn.

This insight underscores the importance of retention-focused strategies and personalized engagement to protect and grow high-value customer segments.

## 8. Conclusion

This analysis explored transactional retail data using SQL to uncover key patterns in revenue generation, customer behavior, and product performance.

The results show that revenue is heavily driven by repeat customers and highly concentrated among a small group of high-value customers. In addition, strong seasonal trends and geographic concentration reveal both risks and opportunities for targeted marketing and strategic expansion.

Overall, this project demonstrates how structured SQL-based analytical workflows can transform raw transactional data into actionable business insights and support data-driven decision-making.

## 9. Limitations and Future Work

This analysis is based solely on historical transactional data and does not incorporate additional factors such as marketing activities, customer demographics, or profit margins.

As a result, the insights are limited to observable purchasing behavior and may not fully capture the underlying drivers of customer decisions.

Future work can extend this analysis by incorporating:
- Customer segmentation techniques (e.g., RFM)
- Churn prediction using machine learning models
- Additional data sources such as marketing campaigns or customer profiles

These enhancements would enable a more comprehensive understanding of customer behavior and support more advanced predictive and prescriptive analytics.